# Coverage and DOP Analysis Using ELFO Constellation
In this notebook, we investigate the number of required satellites to cover the lunar surface for navigation using the ELFO satellites

In [179]:
import numpy as np
import matplotlib.pyplot as plt
import pylupnt as pnt
import os

In [180]:
# options
run_walker_analysis = True
run_elfo_analysis = True

rerun_walker = False
rerun_elfo = False

os.makedirs("output/ex_coverage", exist_ok=True)


## Pattern1: Walker Constellation 

### 1.1 Functions to setup constellation and compute coverage

In [181]:
def walker_constellation(t, p, f, i, h, RE=pnt.R_MOON):
    """
    Setup Walker Constellation

    Args:
        t (int): number of satellites per plane
        p (int): number of planes
        f (int): relative spacing between satellites in the same plane
        i (float): inclination
        h (float): altitude

    Returns:
        coes (np.array): array of COEs for each satellite
    """

    a = RE + h  # semi-major axis [km]
    w = 0  # argument of periapsis [rad]
    e = 0  # eccentricity [-]

    # Initialize list to store orbits
    coes = np.zeros((t*p, 6))

    tot_sat = t * p

    idx = 0
    for plane in range(p):
        for s in range(int(t)):
            alpha = s * 2 * np.pi / t       # intra-plane spacing
            beta = plane * f * 2 * np.pi / tot_sat  # inter-plane spacing
            nu = pnt.wrap2pi(alpha + beta)  # true anomaly
            Omega = pnt.wrap2pi(
                plane * 2 * np.pi / p
            )  # right ascension of the ascending node
            M = pnt.true2eccentric_anomaly(nu, e)  # mean anomaly
            coes[idx] = np.array([a, e, i, Omega, w, M])
            idx += 1

    return coes

For coverage computation, we use Parellel execute to speed up the computation

In [182]:
from joblib import Parallel, delayed
from tqdm import tqdm

def compute_coverage_for_r(i, r, n_sat, lent, rs_pa, min_elevation):
    vis_i = np.zeros((n_sat, lent), dtype=bool)
    for j in range(n_sat):
        az_el_range = pnt.cart2az_el_range(rs_pa[j, :, :3], r)
        vis_i[j] = (az_el_range[..., 1] >= min_elevation)
    return i, np.min(np.sum(vis_i, axis=0))

def compute_coverage(coes, min_elevation, dt, t0, r_surface, dyn, use_tqdm=True):
    T = 2 * np.pi * np.sqrt(coes[0, 0] ** 3 / pnt.GM_MOON)  # [s]
    tspan = t0 + np.arange(0, T, dt)

    n_sat = len(coes)
    lent = len(tspan)

    rv_mci = np.zeros((lent, 6, n_sat))
    rs_pai = np.zeros((n_sat, lent, 6))
    rs_pa = np.zeros((n_sat, lent, 6))
    for si in range(n_sat):
        rv0_pa0 = pnt.classical2cart(coes[si, :], pnt.GM_MOON)  # 
        rv0_mci = pnt.convert_frame(t0, rv0_pa0, pnt.MOON_PA, pnt.MOON_CI)
        rv_mci[:, :, si] = dyn.propagate(rv0_mci, t0, tspan)
        rs_pa[si] = pnt.convert_frame(tspan, rv_mci[:, :, si], pnt.MOON_CI, pnt.MOON_PA)
        rs_pai[si] = pnt.convert_frame(t0 * np.ones(lent), rv_mci[:, :, si], pnt.MOON_CI, pnt.MOON_PA)

    # Prepare arguments for parallelization
    tasks = ((i, r) for i, r in enumerate(r_surface))

    # Use joblib for parallelization
    # n_jobs=-1 uses all available cores; adjust as needed.
    if use_tqdm:
        results = Parallel(n_jobs=-1)(
            delayed(compute_coverage_for_r)(i, r, n_sat, lent, rs_pa, min_elevation) 
            for i, r in tqdm(tasks, total=len(r_surface), desc="Computing coverage")
        )
    else:
        results = Parallel(n_jobs=-1)(
            delayed(compute_coverage_for_r)(i, r, n_sat, lent, rs_pa, min_elevation) 
            for i, r in tasks
        )

    # 'results' will be a list of tuples (i, coverage_val)
    folds_coverage = np.zeros(len(r_surface), dtype=int)
    for i, coverage_val in results:
        folds_coverage[i] = coverage_val

    folds_coverage = folds_coverage.reshape(len(lat), len(lon))

    return rs_pai, rs_pa, folds_coverage

### 1.2 Dynamics
For propagation, we consider perturbations from the sun and Earth

In [183]:
# dynamics for propagation
# Dynamics
dyn_nbody = pnt.NBodyDynamics(pnt.IntegratorType.RKF45)
dyn_nbody.set_integrator_params(pnt.IntegratorParams(max_iter=20, abstol=1e-10, reltol=1e-10))
dyn_nbody.add_body(pnt.Body.Moon(2, 2))
dyn_nbody.add_body(pnt.Body.Earth())
dyn_nbody.add_body(pnt.Body.Sun())
dyn_nbody.set_time_step(5)
dyn_nbody.set_frame(pnt.MOON_CI)

### 1.3 Surface Mesh
Create a surface mesh of evaluation points

In [ ]:
# Surface mesh
lat = np.linspace(-90, 90, 181) * pnt.RAD
lon = np.linspace(0, 360, 361) * pnt.RAD
lats_mesh, lons_mesh = np.meshgrid(lat, lon)
lats = lats_mesh.flatten()
lons = lons_mesh.flatten()
alts = np.zeros_like(lats)
r_surface = pnt.lat_lon_alt2cart(np.array([lats, lons, alts]).T, pnt.R_MOON)

print(lats_mesh.shape, lons_mesh.shape, r_surface.shape)

### 1.4 Run Single Example Case (Walker)

In [ ]:
# run coverage simulation
dt = 60*5  # interval for evaluation [s]
t0 = pnt.gregorian2time(2025, 1, 1, 12, 0, 0)  # initial time [s]
min_elevation = 10 * pnt.RAD  # [rad] Minimum elevation angle

coe = walker_constellation(t=3, p=6, f=1, i=55 * pnt.RAD, h=5000, RE=pnt.R_MOON)
rv_pai, rs_pa, coverage = compute_coverage(coe, min_elevation, dt, t0, r_surface, dyn_nbody)

In [ ]:
# plot constellation
import plotly.graph_objects as go 

n_sat = len(coe)
orb_plot_mci = np.zeros((n_sat, rv_pai.shape[1], 6))
orb_plot_pa = np.zeros((n_sat, rv_pai.shape[1], 6))
for i in range(len(coe)):
    orb_plot_mci[i] = rv_pai[i]
    orb_plot_pa[i] = rs_pa[i]

# In MCI frame
fig = go.Figure()
pnt.plot.plot_body(fig, pnt.MOON)
pnt.plot.plot_orbits(
    fig,
    orb_plot_mci,
    color = pnt.plot.sample_colorscale("rainbow", np.linspace(0, 1, n_sat)),
)
pnt.plot.set_view(fig, azimuth=-130, elevation=10, zoom=2.5)
fig.update_layout(width=400, height=400)
fig.show()

# In PA frame
fig = go.Figure()
pnt.plot.plot_body(fig, pnt.MOON)
pnt.plot.plot_orbits(
    fig,
    orb_plot_pa,
    color = pnt.plot.sample_colorscale("rainbow", np.linspace(0, 1, n_sat)),
)
pnt.plot.set_view(fig, azimuth=-130, elevation=10, zoom=2.5)
fig.update_layout(width=700, height=700)
fig.show()

In [187]:
from matplotlib.colors import ListedColormap

def plot_coverage(folds_coverage, rv_pai):
    fig = plt.figure(figsize=(8, 4))
    plt.xlim(0, 360)
    plt.ylim(-90, 90)
    plt.xlabel("Longitude [deg]")
    plt.ylabel("Latitude [deg]")

    max_coverage = np.max(np.max(folds_coverage))

    cov_cmap_tmp = plt.get_cmap("Blues", max_coverage + 1)
    cov_cmap = ListedColormap(cov_cmap_tmp([i for i in range(max_coverage)]))

    n_sat = rv_pai.shape[0]

    for si in range(n_sat):
        rv_pa = rv_pai[si]
        az = np.arctan2(rv_pa[:, 1], rv_pa[:, 0]) + np.pi
        el = np.arcsin(rv_pa[:, 2] / np.linalg.norm(rv_pa[:, :3], axis=1))
        plt.plot(az * pnt.DEG, el * pnt.DEG, 'ko', markersize=1)
        plt.plot(az[0] * pnt.DEG, el[0] * pnt.DEG, 'ro', markersize=5)

    mat = plt.pcolormesh(
        lons_mesh * pnt.DEG,
        lats_mesh * pnt.DEG,
        folds_coverage.T,
        cmap=cov_cmap,
        vmin=-0.5,
        vmax=max_coverage - 0.5,
        shading='auto',
        alpha=0.9,
    )
    plt.colorbar(mat, ticks=np.arange(max_coverage))
    plt.title(f"Folds of Coverage for {pnt.time2gregorian_string(t0)} TAI")
    plt.grid()
    plt.tight_layout()
    plt.show()

    return fig

In [ ]:
print("Coverage:", coverage.shape)
plot_coverage(coverage, rv_pai)

In [ ]:
# compute mean coverage per latitude band
mean_coverage = np.mean(coverage, axis=1)
plt.plot(lat * pnt.DEG, mean_coverage)
plt.xlabel("Latitude [deg]")
plt.ylabel("Mean coverage")
plt.grid()
plt.show()

## Change Altitude and Inclination and Compute Coverage

In [ ]:
if run_walker_analysis:
    # run coverage simulation
    dt = 60*5  # interval for evaluation [s]
    t0 = pnt.gregorian2time(2025, 1, 1, 12, 0, 0)  # initial time [s]
    min_elevation = 10 * pnt.RAD  # [rad] Minimum elevation angle

    t = 3  # number of satellites per plane
    p = 6  # number of planes
    f = 1  # relative spacing between satellites in the same plane
    config = (t, p, f)

    # dynamics for propagation
    dyn_tb = pnt.NBodyDynamics(pnt.IntegratorType.RKF45)
    dyn_tb.set_integrator_params(pnt.IntegratorParams(max_iter=20, abstol=1e-10, reltol=1e-10))
    dyn_tb.add_body(pnt.Body.Moon(2, 0))
    dyn_tb.set_time_step(5)
    dyn_tb.set_frame(pnt.MOON_CI)

    if config == (6, 3, 1):
        hlist = np.arange(3500, 5500, 200)
        ilist = np.arange(40, 70, 5)
    elif config == (3, 6, 1):
        hlist = np.arange(5500, 7500, 200)
        ilist = np.arange(40, 70, 5)
    elif config == (5, 4, 1):
        hlist = np.arange(4500, 6500, 200)
        ilist = np.arange(40, 70, 5)
    elif config == (4, 5, 1):
        hlist = np.arange(4000, 6000, 200)
        ilist = np.arange(40, 70, 5)
    else:
        raise ValueError("Unsupported configuration")

    # 
    runsim = False

    if rerun_walker or not os.path.exists(f"output/ex_coverage/walker_constellation_{t}_{p}_{f}.npz"):
        runsim = True

    if runsim:
        min_alt = np.inf
        max_cov_incs = []

        # Surface mesh
        total_idx = len(hlist) * len(ilist)
        total_coverage_over4 = np.zeros((len(hlist), len(ilist)))

        for i, h in enumerate(hlist):
            for j, inc in enumerate(ilist):
                sim_idx = i * len(ilist) + j

                print(f"sim_idx: {sim_idx}/{total_idx} i: {i}, j: {j}")
                coe = walker_constellation(t, p, f, i=inc * pnt.RAD, h=h, RE=pnt.R_MOON)
                rv_pai, rs_pa, coverage = compute_coverage(coe, min_elevation, dt, t0, r_surface, dyn_tb, use_tqdm=False)
                mean_coverage = np.mean(coverage, axis=1)
                over4_coverage = np.sum(coverage >= 4, axis=1) / coverage.shape[1]
                total_coverage_over4[i, j] = np.sum(np.sum(coverage >= 4, axis=0)) / coverage.shape[0] / coverage.shape[1] * 100

                if total_coverage_over4[i, j] >= 100:
                    full_coverage = True
                else:
                    full_coverage = False

                if full_coverage and min_alt > h:
                    min_alt = h
                    max_cov_incs = [inc]
                elif full_coverage and min_alt == h:
                    max_cov_incs.append(inc)

        # save results
        np.savez(
            f"output/ex_coverage/walker_constellation_{t}_{p}_{f}.npz",
            hlist=hlist,
            ilist=ilist,
            total_coverage_over4=total_coverage_over4,
            min_alt=min_alt,
            max_cov_incs=max_cov_incs,
        )

        print("min_alt:", min_alt)
        print("max_cov_incs:", max_cov_incs)


In [ ]:
t = 3  # number of satellites per plane
p = 6  # number of planes
f = 1  # relative spacing between satellites in the same plane

# load results
if run_walker_analysis:
    data = np.load(f"output/ex_coverage/walker_constellation_{t}_{p}_{f}.npz")
    hlist = data["hlist"]
    ilist = data["ilist"]
    min_alt = data["min_alt"]
    max_cov_incs = data["max_cov_incs"]
    total_coverage_over4 = data["total_coverage_over4"]

    # plot results
    fig = plt.figure(figsize=(8, 4))

    for j, inc in enumerate(ilist):
        plt.plot(hlist, total_coverage_over4[:, j], 'o-', label=f"inc = {inc} deg")
    plt.xlabel("Altitude [km]")
    plt.ylabel("Fraction of time with 4+ coverage [%]")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()


    # 2d contour plot
    fig = plt.figure(figsize=(10, 6))

    igrid, hgrid = np.meshgrid(ilist, hlist)

    plt.contourf(ilist, hlist, total_coverage_over4, cmap="viridis", levels=50, alpha=0.3)
    plt.scatter(igrid, hgrid, c=total_coverage_over4,  cmap="viridis")
    # print the text of coverage
    for i in range(len(hlist)):
        for j in range(len(ilist)):
            eps_h = 10
            eps_i = 1
            plt.text(ilist[j] + eps_i, hlist[i] + eps_h, f"{total_coverage_over4[i, j]:.2f}", ha='center', va='center', fontsize=9)

    # plot the minimum altitude point
    plt.scatter(max_cov_incs, [min_alt] * len(max_cov_incs), c="red", marker="x", label="Minimum altitude")
    plt.colorbar()
    plt.xlabel("Inclination [deg]")
    plt.ylabel("Altitude [km]")
    # set minor grid to hlist and ilist
    plt.grid()
    plt.xticks(ilist)
    plt.yticks(hlist)
    plt.ylim(hlist[0] - 50, hlist[-1] + 50)
    plt.xlim(ilist[0] - 0.5, ilist[-1] + 0.5)
    plt.title("Ratio with 4+ coverage [%] \n (Walker with Planes = {}, Satellite Per Plane = {})".format(p, t))

    plt.tight_layout()
    plt.savefig("output/ex_coverage/walker_constellation_{}_{}_{}.png".format(t, p, f))

    plt.show()


# Pattern 2: ELFO Constellation

### 2.1 eccentricity vs omega analysis
First, we analyze the relationship between eccentricity and w, to make the orbit frozen

In [ ]:
# e vs w
inc = np.linspace(0, 75, 100) * pnt.RAD
e = np.sqrt(1 - 5/3 * np.cos(inc) * np.cos(inc))
a_min = pnt.R_MOON * np.ones(e.shape) / (np.ones(e.shape) - e)

# convert from op frame to pa frame


plt.figure()
plt.plot(inc * pnt.DEG, e, 'o-', color='blue', label="Eccentricity")
plt.grid(True)
plt.xlabel("Inclination [deg]")
plt.ylabel("Eccentricity")
plt.legend()

# second axis
plt.twinx()
plt.plot(inc * pnt.DEG, a_min - pnt.R_MOON, 'o-', color='red', label="Minimum altitude")
plt.ylabel("Minimum seminmajor axis [km]")
plt.legend()

plt.show()

### 2.2 Function Definitions
Next, we define functions to compute elfo constellations and its plot

In [193]:
def elfo_constellation(t, p, f, inc, h, RE=pnt.R_MOON, add_equatorial=False, eqsat_num=None):
    """
    Setup Walker Constellation

    Args:
        t (int): number of satellites per plane
        p (int): number of planes
        f (int): relative spacing between satellites in the same plane
        i (float): inclination
        h (float): altitude

    Returns:
        coes (np.array): array of COEs for each satellite
    """
    # Initialize list to store orbits
    n_sat = t * p * 2
    if add_equatorial and eqsat_num is not None:
        n_sat += eqsat_num

    coes = np.zeros((n_sat, 6))

    a = RE + h  # semi-major axis [km]
    e2 = 1 - 5/3 * np.cos(inc) * np.cos(inc)
    if e2 >= 0:
        e = np.sqrt(e2)
    else:
        return coes, False
    
    a_peri = (h + RE) * (1 - e)
    if a_peri < RE:
        return coes, False

    w1 = 90 * pnt.RAD  # argument of periapsis [rad]
    w2 = 270 * pnt.RAD  # argument of periapsis [rad]

    tot_sat = t * p * 2 # total number of satellites

    idx = 0
    for plane in range(p):
        for s in range(int(t)):
            alpha = s * 2 * np.pi / t       # intra-plane spacing
            beta1 = plane * f * 2 * np.pi / tot_sat  # inter-plane spacing
            beta2 = plane * f * 2 * np.pi / tot_sat  # inter-plane spacing
            phase1 = alpha + beta1  # w = 90
            phase2 = alpha + beta2  # w = 270
            nu1 = pnt.wrap2pi(phase1)  # true anomaly
            nu2 = pnt.wrap2pi(phase2)  # true anomaly
            Omega = pnt.wrap2pi(
                plane * 2 * np.pi / p
            )  # right ascension of the ascending node
            M1 = pnt.true2eccentric_anomaly(nu1, e)  # mean anomaly
            M2 = pnt.true2eccentric_anomaly(nu2, e)

            coes[idx] = np.array([a, e, inc, Omega, w1, M1])
            idx += 1
            coes[idx] = np.array([a, e, inc, Omega, w2, M2])
            idx += 1

    # all the coes are in op frame, convert to pa frame
    for si in range(len(coes)):
        rv0_op0 = pnt.classical2cart(coes[si, :], pnt.GM_MOON)  # 
        rv0_pa0 = pnt.convert_frame(t0, rv0_op0, pnt.MOON_OP, pnt.MOON_PA)
        coes[si] = pnt.cart2classical(rv0_pa0, pnt.GM_MOON)

    if add_equatorial and eqsat_num is not None:
        a = RE + h  # semi-major axis [km]
        theta = np.linspace(0, 2*np.pi, eqsat_num, endpoint=False)
        for i in range(eqsat_num):
            coes[idx] = np.array([a, 0, 0, 0, 0, theta[i]])
            idx += 1

    return coes, True

In [194]:
# plot constellation
import plotly.graph_objects as go 

def plot_constellation(coe, rv_pai, savedir=None):
    n_sat = len(coe)
    orb_plot_mci = np.zeros((n_sat, rv_pai.shape[1], 6))
    orb_plot_pa = np.zeros((n_sat, rv_pai.shape[1], 6))
    for i in range(len(coe)):
        orb_plot_mci[i] = rv_pai[i]
        orb_plot_pa[i] = rs_pa[i]

    # In PA frame
    fig = go.Figure()
    pnt.plot.plot_body(fig, pnt.MOON)
    pnt.plot.plot_orbits(
        fig,
        orb_plot_pa,
        color = pnt.plot.sample_colorscale("rainbow", np.linspace(0, 1, n_sat)),
    )
    pnt.plot.set_view(fig, azimuth=-130, elevation=10, zoom=2.5)
    fig.update_layout(width=600, height=600)

    if savedir is not None:
        fig.write_image(savedir)

    fig.show()

    return fig

define dynamics to propagate orbits

In [195]:
# dynamics for propagation
dyn_tb = pnt.NBodyDynamics(pnt.IntegratorType.RKF45)
dyn_tb.set_integrator_params(pnt.IntegratorParams(max_iter=20, abstol=1e-10, reltol=1e-10))
dyn_tb.add_body(pnt.Body.Moon(2, 0))
dyn_tb.set_time_step(5)
dyn_tb.set_frame(pnt.MOON_CI)

### 2.3 Run Single Case Analysis (ELFO)

In [ ]:
# example constellation
# (t, p, h, inc) = (4, 3, 9500, 55)
dt = 60*5  # interval for evaluation [s]
t0 = pnt.gregorian2time(2025, 1, 1, 12, 0, 0)  # initial time [s]
min_elevation = 10 * pnt.RAD  # [rad] Minimum elevation angle

t = 6  # number of satellites per plane
p = 2  # number of planes / 2
f = 1  # relative spacing between satellites in the same plane

coe_pa0, valid = elfo_constellation(t, p, f, 45 * pnt.RAD, h=9000 - pnt.R_MOON, RE=pnt.R_MOON, add_equatorial=False, eqsat_num=4)
rv_pai, rs_pa, coverage = compute_coverage(coe_pa0, min_elevation, dt, t0, r_surface, dyn_tb, use_tqdm=False)


In [ ]:
fig = plot_constellation(coe_pa0, rv_pai, savedir="output/ex_coverage/elfo_constellation_orbit_{}_{}_{}.png".format(t, p, f))

In [ ]:
# coverage plot
print("Coverage:", coverage.shape)
plot_coverage(coverage, rv_pai)

mean_coverage = np.mean(coverage, axis=1)
over4_coverage = np.sum(coverage >= 4, axis=1) / coverage.shape[1]

fig, axes = plt.subplots(2, 1, figsize=(8, 8))

axes[0].plot(lat * pnt.DEG, mean_coverage, 'o-', label="mean coverage")
axes[0].set_xlabel("Latitude [deg]")
axes[0].set_ylabel("Mean coverage")
axes[0].grid()

axes[1].plot(lat * pnt.DEG, over4_coverage, 'o-', label="coverage > 4")
axes[1].set_xlabel("Latitude [deg]")
axes[1].set_ylabel("Coverage > 4")
axes[1].grid()

### 2.4 Run Coverage Simulation with Different Altitudes and Inclinations

In [ ]:
# run coverage simulation
dt = 60*5  # interval for evaluation [s]
t0 = pnt.gregorian2time(2025, 1, 1, 12, 0, 0)  # initial time [s]
min_elevation = 10 * pnt.RAD  # [rad] Minimum elevation angle

t = 4  # number of satellites per plane
p = 3  # number of planes / 2  (total planes = 2*p because w = 90 and 270)
f = 1  # relative spacing between satellites in the same plane
config = (t, p, f)

# dynamics for propagation
dyn_tb = pnt.NBodyDynamics(pnt.IntegratorType.RKF45)
dyn_tb.set_integrator_params(pnt.IntegratorParams(max_iter=20, abstol=1e-10, reltol=1e-10))
dyn_tb.add_body(pnt.Body.Moon(2, 0))
dyn_tb.set_time_step(5)
dyn_tb.set_frame(pnt.MOON_CI)

# altitude range
hlist = np.arange(3000, 13000, 1000) - pnt.R_MOON  # altitude range
ilist = np.arange(40, 70, 5)  # inclination range

# altitude and inclination range
min_alt = np.inf
max_cov_incs = []

# Surface mesh
total_idx = len(hlist) * len(ilist)
total_coverage_over4 = np.zeros((len(hlist), len(ilist)))

for i, h in enumerate(hlist):
    for j, inc in enumerate(ilist):
        sim_idx = i * len(ilist) + j

        # compute coverage
        coe_pa0, valid = elfo_constellation(t, p, f, inc * pnt.RAD, h=h, RE=pnt.R_MOON, add_equatorial=False, eqsat_num=4)

        e2 = 1 - 5/3 * np.cos(inc * pnt.RAD) * np.cos(inc * pnt.RAD)
        if e2 >= 0:
            e = np.sqrt(e2)
            altmin = (h + pnt.R_MOON) * (1 - e) - pnt.R_MOON
        else:
            continue

        if not valid:
            print(f"sim_idx: {sim_idx}/{total_idx} inc: {inc}, h: {h}, valid: {valid}, coverage: 0%")
            continue
        
        rv_pai, rs_pa, coverage = compute_coverage(coe_pa0, min_elevation, dt, t0, r_surface, dyn_tb, use_tqdm=False)

        mean_coverage = np.mean(coverage, axis=1)
        over4_coverage = np.sum(coverage >= 4, axis=1) / coverage.shape[1]
        total_coverage_over4[i, j] = np.sum(np.sum(coverage >= 4, axis=0)) / coverage.shape[0] / coverage.shape[1] * 100

        print(f"sim_idx: {sim_idx}/{total_idx} inc: {inc}, h: {h}, valid: {valid}, coverage: {total_coverage_over4[i, j]:.1f}%")

        if total_coverage_over4[i, j] >= 100:
            full_coverage = True
        else:
            full_coverage = False

        if full_coverage and min_alt > h:
            min_alt = h
            max_cov_incs = [inc]
        elif full_coverage and min_alt == h:
            max_cov_incs.append(inc)

# save results
np.savez(
    f"output/ex_coverage/elfo_constellation_{t}_{p}_{f}.npz",
    hlist=hlist,
    ilist=ilist,
    total_coverage_over4=total_coverage_over4,
    min_alt=min_alt,
    max_cov_incs=max_cov_incs,
)

print("min_alt:", min_alt)
print("max_cov_incs:", max_cov_incs)

In [ ]:
t = 4  # number of satellites per plane
p = 3  # number of planes / 2
f = 1  # relative spacing between satellites in the same plane

# load results
data = np.load(f"output/ex_coverage/elfo_constellation_{t}_{p}_{f}.npz")
hlist = data["hlist"]
ilist = data["ilist"]
min_alt = data["min_alt"]
max_cov_incs = data["max_cov_incs"]
total_coverage_over4 = data["total_coverage_over4"]

alist = hlist + pnt.R_MOON
total_coverage_over4[total_coverage_over4 == 0] = np.nan

# plot results
fig = plt.figure(figsize=(8, 4))

for j, inc in enumerate(ilist):
    plt.plot(alist, total_coverage_over4[:, j], 'o-', label=f"inc = {inc} deg")
plt.xlabel("Altitude [km]")
plt.ylabel("Fraction of time with 4+ coverage [%]")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.ylim(80, 101)
plt.xlim(alist[0] - 500, alist[-1] + 500)
plt.show()


# 2d contour plot
fig = plt.figure(figsize=(10, 6))
igrid, agrid = np.meshgrid(ilist, alist)

plt.contourf(ilist, alist, total_coverage_over4, cmap="viridis", levels=50, alpha=0.3)
plt.scatter(igrid, agrid, c=total_coverage_over4,  cmap="viridis")
# print the text of coverage
for i in range(len(alist)):
    for j in range(len(ilist)):
        eps_h = 10
        eps_i = 1
        if total_coverage_over4[i, j] > 0:
            plt.text(ilist[j] + eps_i, alist[i] + eps_h, f"{total_coverage_over4[i, j]:.2f}", ha='center', va='center', fontsize=9)

# plot the minimum altitude point
plt.scatter(max_cov_incs, [min_alt + pnt.R_MOON] * len(max_cov_incs), c="red", marker="x", label="Minimum altitude")
plt.colorbar()
plt.xlabel("Inclination [deg]")
plt.ylabel("Semi-major axis [km]")
# set minor grid to hlist and ilist
plt.grid()
plt.xticks(ilist)
plt.yticks(alist)
plt.ylim(pnt.R_MOON, alist[-1] + 50)
plt.xlim(ilist[0] - 0.5, ilist[-1] + 0.5)
plt.title("Ratio with 4+ coverage [%] \n (ELFO with Planes = {}, Satellite Per Plane = {})".format(p, t))

# plot the minimum altitude
mina_inc = np.zeros(ilist.shape)
for j, inc in enumerate(ilist):
    e2 = 1 - 5/3 * np.cos(inc * pnt.RAD) * np.cos(inc * pnt.RAD)
    if e2 >= 0:
        e = np.sqrt(e2)
        amin = pnt.R_MOON / (1 - e)
        mina_inc[j] = amin
        
plt.plot(ilist, mina_inc, 'o--', color='red')
plt.text(53, 4500, 'periapsis > moon radius', color='red')

plt.tight_layout()
plt.savefig("output/ex_coverage/elfo_constellation_{}_{}_{}.png".format(t, p, f))

plt.show()